# Session 25 — Semantic Layering & Metadata Preparation

**Dataset:** Online Retail (UK online gift retailer, Dec-2010 → Dec-2011), 1 % sample, ~5.4 k invoice lines.

### What you will learn
| # | Concept | What you will *see* in this notebook |
|---|---------|--------------------------------------|
| 1 | **Metadata** | Technical (schema, location, format), business (comments, owners, definitions) and operational (statistics, data-quality counts) metadata attached to tables |
| 2 | **Metastore / Catalog** | Where Spark keeps that metadata, how to query it, and proof that it survives a restart |
| 3 | **Managed vs. unmanaged (external) tables** | Who owns the files, what `DROP` / `TRUNCATE` / `RENAME` really do, and why partitions can "disappear" |
| 4 | **Semantic layering** | Bronze → Silver → Gold layers, and a Gold *semantic layer* of business-friendly dimensions, facts and governed metrics |

### Agenda
0. Big picture & setup
1. Load the data, temp views (session-only metadata)
2. The metastore & catalog
3. Managed vs. external tables — four experiments
4. Bronze layer — external table over raw files (schema-on-read)
5. Silver layer — cleaned, typed, documented managed table
6. Metadata deep dive — statistics and the optimizer
7. Gold layer — the semantic layer
8. Metadata as data — generating a data dictionary
9. Wrap-up & proof of persistence
10. Exercises

> **Requirements:** Spark 3.5+ (pip `pyspark` already ships with Hive support). The CSV is expected at `data/online-retail-dataset-one-percent-sample.csv` relative to the notebook.

## Part 0 — The big picture

When you run `SELECT * FROM silver.sales_lines`, Spark has to answer two very different questions:

1. **"What is `silver.sales_lines`?"** — which columns, types, file format, where the files are, how it is partitioned, how big it is. That is **metadata**, and it lives in the **metastore**.
2. **"Give me the bytes."** — that is **data**, and it lives in **storage** (local disk here; S3 / ADLS / GCS / HDFS in production).

```
                 SQL / DataFrame API
                         │
              ┌──────────▼───────────┐
              │  Spark driver        │
              │  parser → analyzer → │   asks the CATALOG: "what & where is it?"
              │  optimizer → planner │──────────────────────┐
              └──────────┬───────────┘                      │
                         │ reads/writes bytes               ▼
    ┌────────────────────▼──────────────────┐   ┌──────────────────────────────┐
    │  STORAGE  (files)                     │   │  METASTORE  (metadata)       │
    │  spark-warehouse/   ← MANAGED tables  │   │  databases, tables, columns, │
    │  external-data/     ← EXTERNAL tables │   │  types, format, location,    │
    │  landing/           ← raw source files│   │  partitions, statistics,     │
    │                                       │   │  comments, properties,       │
    │                                       │   │  view SQL text               │
    └───────────────────────────────────────┘   │  here: ./metastore_db (Derby)│
                                                └──────────────────────────────┘
```

**Key vocabulary**

* **Metadata** — data about data. We will use three flavours:
  * *Technical*: schema, data types, file format, location, partitions.
  * *Business*: descriptions, owners, stewards, definitions, sensitivity tags.
  * *Operational*: row counts, sizes, column statistics, load timestamps, data-quality results.
* **Metastore** — the database that *stores* metadata. Locally Spark uses an embedded Apache Derby database (`./metastore_db`). In production it is a Hive Metastore backed by MySQL/Postgres, AWS Glue Data Catalog, Databricks Unity Catalog, or an Iceberg REST catalog (e.g. Polaris).
* **Catalog** — the API/namespace through which Spark *reads* the metastore (`spark.catalog`, `SHOW TABLES`, `DESCRIBE …`). Objects are addressed as `catalog.database.table`.
* **Managed table** — Spark owns **both** metadata **and** data. Files live in the warehouse directory. `DROP TABLE` deletes the files.
* **External (unmanaged) table** — Spark owns **only** the metadata; it points at files at a `LOCATION` you control. `DROP TABLE` deletes only the metadata.
* **Semantic layer** — a business-facing layer (views, dimensions, facts, metric definitions) that translates physical tables into the language of the business, so everybody computes "revenue" the same way.

### 0.1 Start Spark *with a persistent metastore*

Two changes from a plain session:

* `.enableHiveSupport()` switches the catalog from `in-memory` (metadata vanishes when the session ends) to `hive` (metadata is persisted in `./metastore_db`).
* `spark.sql.warehouse.dir` pins where **managed** tables are stored.

> ⚠️ These are *static* settings — they only apply when the session is first created. If a Spark session is already running in this kernel, restart the kernel first.

In [1]:
import pyspark, os, sys, shutil, pandas as pd
from datetime import datetime
from pathlib import Path
from urllib.parse import urlparse
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import pyspark.sql.types as T
os.environ['PYSPARK_PYTHON'] = os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
print("Python", sys.executable)
print("Java", os.environ["JAVA_HOME"])

PROJECT_DIR   = Path(os.getcwd())
WAREHOUSE_DIR = PROJECT_DIR / "spark-warehouse"   # Spark decides layout here  -> MANAGED tables
EXTERNAL_DIR  = PROJECT_DIR / "external-data"     # storage that WE control     -> EXTERNAL tables
LANDING_DIR   = PROJECT_DIR / "landing"           # raw files as delivered by the source system

METASTORE_DB = PROJECT_DIR / "metastore_db"

spark = (SparkSession.builder
         .appName("semantic-layering-and-meta-data-preparation")
         .config("spark.sql.warehouse.dir", WAREHOUSE_DIR.as_posix())
         .config("spark.sql.warehouse.dir", "/path/to/course/spark-warehouse")
         .config("spark.hadoop.javax.jdo.option.ConnectionURL",
                 f"jdbc:derby:;databaseName={METASTORE_DB};create=true")
         .enableHiveSupport()          # <- persistent metastore (embedded Derby in ./metastore_db)
         .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("WARN")

print("Spark version          :", spark.version)
print("Catalog implementation :", spark.conf.get("spark.sql.catalogImplementation"))   # expect: hive
print("Warehouse directory    :", spark.conf.get("spark.sql.warehouse.dir"))
spark

Python /usr/local/bin/python3.12
Java /usr/lib/jvm/java-21-openjdk-arm64


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/usr/local/lib/python3.12/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/11/10 15:01:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version          : 4.3.0
Catalog implementation : hive
Warehouse directory    : file:/path/to/course/spark-warehouse


### 0.2 Helper functions
Small utilities we will reuse to look at **metadata** (`table_summary`) and at **the files on disk** (`tree`) side by side.

In [2]:
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.max_rows", 200)

def sql(query, n=100):
    """Run Spark SQL and return the first n rows as a pandas DataFrame (pretty notebook display)."""
    return spark.sql(query).limit(n).toPandas()

def catalog_tables(db=None):
    """List tables/views registered in the catalog as a pandas DataFrame."""
    # note: listTables(db) also returns *local temp views*; we hide them when a db is given
    tables = spark.catalog.listTables(db) if db else spark.catalog.listTables()
    if db:
        tables = [t for t in tables if not t.isTemporary]
    return pd.DataFrame([t._asdict() for t in tables])

def table_summary(fqtn, keys=("Type", "Provider", "Location", "Comment", "Partition Provider",
                              "Table Properties", "Statistics", "View Text")):
    """Pick the interesting rows out of DESCRIBE TABLE EXTENDED."""
    rows = spark.sql(f"DESCRIBE TABLE EXTENDED {fqtn}").collect()
    info = {}
    for r in rows:
        name = (r.col_name or "").strip()
        if name in keys and name not in info:
            info[name] = r.data_type
    return pd.DataFrame({"property": list(info), "value": list(info.values())}).set_index("property")

def table_location(fqtn):
    """Physical location of a table, read from the metastore."""
    rows = spark.sql(f"DESCRIBE TABLE EXTENDED {fqtn}").collect()
    loc = next((r.data_type for r in rows if r.col_name == "Location"), None)
    return Path(urlparse(loc).path) if loc else None

def tree(path, max_files=3):
    """Print a directory tree (hides _SUCCESS and .crc housekeeping files). max_files=0 -> folders only."""
    path = Path(path)
    if not path.exists():
        print(f"{path}   <-- DOES NOT EXIST")
        return
    for root, dirs, files in os.walk(path):
        dirs.sort()
        depth = len(Path(root).relative_to(path).parts)
        indent = "    " * depth
        print(f"{indent}{Path(root).name}/")
        files = sorted(f for f in files if not f.startswith((".", "_")))
        for f in files[:max_files]:
            print(f"{indent}    {f}  ({(Path(root) / f).stat().st_size:,} bytes)")
        if max_files and len(files) > max_files:
            print(f"{indent}    ... {len(files) - max_files} more file(s)")

def dir_size(path):
    path = Path(path) if path else None
    return sum(f.stat().st_size for f in path.rglob("*") if f.is_file()) if path and path.exists() else 0

### 0.3 Clean slate (makes the notebook re-runnable)
Drops the demo databases and deletes the demo folders. **Skip this cell** in Part 9 when we prove that metadata persists across restarts.

In [3]:
DEMO_DATABASES = ["lab", "bronze", "silver", "gold", "governance"]
RESET = True

if RESET:
    for db in DEMO_DATABASES:
        spark.sql(f"DROP DATABASE IF EXISTS {db} CASCADE")
        shutil.rmtree(WAREHOUSE_DIR / f"{db}.db", ignore_errors=True)
    shutil.rmtree(EXTERNAL_DIR, ignore_errors=True)
    shutil.rmtree(LANDING_DIR, ignore_errors=True)

print("Databases now in the metastore:", [d.name for d in spark.catalog.listDatabases()])

26/11/10 15:01:12 WARN TxnHandler: Cannot perform cleanup since metastore table does not exist
26/11/10 15:01:12 WARN TxnHandler: Cannot perform cleanup since metastore table does not exist
26/11/10 15:01:12 WARN TxnHandler: Cannot perform cleanup since metastore table does not exist
26/11/10 15:01:12 WARN TxnHandler: Cannot perform cleanup since metastore table does not exist


Databases now in the metastore: ['default']


## Part 1 — Load the data

`inferSchema` makes Spark scan the file and *guess* the types. The guessed schema is the first piece of **technical metadata** we meet. Notice the guesses: `CustomerID` becomes a `double` (because of values like `16906.0`) and `InvoiceDate` stays a `string`. We will fix both deliberately in the Silver layer.

In [4]:
file_name = os.path.join(os.getcwd(), "data/online-retail-dataset-one-percent-sample.csv")
df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(file_name)
df

DataFrame[InvoiceNo: string, StockCode: string, Description: string, Quantity: int, InvoiceDate: string, UnitPrice: double, CustomerID: double, Country: string]

In [5]:
df.printSchema()

root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: string (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: double (nullable = true)
 |-- Country: string (nullable = true)



In [6]:
df.show(3)

+---------+---------+--------------------+--------+---------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity|    InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+---------------+---------+----------+--------------+
|   567288|    22098|BOUDOIR SQUARE TI...|       4|9/19/2011 11:49|     0.39|   16906.0|United Kingdom|
|   541827|   85199S|SMALL HANGING IVO...|       1|1/21/2011 17:05|     0.83|      NULL|United Kingdom|
|   574740|    22027|TEA PARTY BIRTHDA...|      24|11/6/2011 16:07|     0.42|   12357.0|   Switzerland|
+---------+---------+--------------------+--------+---------------+---------+----------+--------------+
only showing top 3 rows


In [7]:
print("rows:", df.count())
# Schema is metadata you can inspect programmatically, too
[(f.name, f.dataType.simpleString(), f.nullable) for f in df.schema.fields]

rows: 5419


[('InvoiceNo', 'string', True),
 ('StockCode', 'string', True),
 ('Description', 'string', True),
 ('Quantity', 'int', True),
 ('InvoiceDate', 'string', True),
 ('UnitPrice', 'double', True),
 ('CustomerID', 'double', True),
 ('Country', 'string', True)]

### 1.1 Temp views: metadata that lives only in your session

A **temporary view** is just a *name* mapped to a *query plan*, held in the memory of this `SparkSession`.
* No files are written.
* Nothing goes to the metastore → it disappears when the session ends.
* Other sessions cannot see it (a *global* temp view is visible to other sessions of the same application, under the special `global_temp` database).

In [8]:
df.createOrReplaceTempView("sales")

# tableType = TEMPORARY, isTemporary = True, namespace is empty: it does not belong to any database
catalog_tables()

,name,catalog,namespace,description,tableType,isTemporary
0,sales,None,[],None,TEMPORARY,True


In [9]:
sql("""
SELECT Country,
       COUNT(*)                          AS line_count,
       ROUND(SUM(Quantity * UnitPrice), 2) AS naive_revenue
FROM   sales
GROUP  BY Country
ORDER  BY naive_revenue DESC
""", 8)

/usr/local/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:324: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/usr/local/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:374: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


,Country,line_count,naive_revenue
0,United Kingdom,4972,98270.94
1,Netherlands,26,2439.75
2,EIRE,75,2238.73
3,Germany,96,2055.22
4,France,91,1784.18
5,Australia,8,482.08
6,Spain,22,367.86
7,Belgium,21,360.14


In [10]:
df.createOrReplaceGlobalTempView("sales_global")

other_session = spark.newSession()          # same application, different session
try:
    other_session.sql("SELECT COUNT(*) FROM sales").show()
except Exception as e:
    print("Local temp view from another session ->", type(e).__name__, ":", str(e).splitlines()[0][:120])

print("Global temp view from another session ->", other_session.sql("SELECT COUNT(*) AS n FROM global_temp.sales_global").first().n, "rows")

{"ts": "2026-11-10 15:01:14.675", "level": "ERROR", "logger": "SQLQueryContextLogger", "msg": "[TABLE_OR_VIEW_NOT_FOUND] The table or view `sales` cannot be found. Verify the spelling and correctness of the schema and catalog.\nSearch path: [`system`.`builtin`, `system`.`session`, `spark_catalog`.`default`].\nIf you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.\nTo tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS. SQLSTATE: 42P01", "context": {"errorClass": "TABLE_OR_VIEW_NOT_FOUND"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o89.sql.\n: org.apache.spark.sql.AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `sales` cannot be found. Verify the spelling and correctness of the schema and catalog.\nSearch path: [`system`.`builtin`, `system`.`session`, `spark_catalog`.`default`].\nIf you did not qualify the name with a schema, ve

Local temp view from another session -> AnalysisException : [TABLE_OR_VIEW_NOT_FOUND] The table or view `sales` cannot be found. Verify the spelling and correctness of the schema a
Global temp view from another session -> 5419 rows


## Part 2 — The metastore & catalog

The catalog is a hierarchy: **catalog → database (a.k.a. schema / namespace) → table / view → column**.
Spark's built-in catalog is called `spark_catalog`; `default` is the database you start in.

In [11]:
print("Catalog implementation:", spark.conf.get("spark.sql.catalogImplementation"))
print("Current catalog       :", spark.catalog.currentCatalog())
print("Current database      :", spark.catalog.currentDatabase())
pd.DataFrame([d._asdict() for d in spark.catalog.listDatabases()])

Catalog implementation: hive
Current catalog       : spark_catalog
Current database      : default


,name,catalog,description,locationUri
0,default,spark_catalog,Default Hive database,file:/app/work/data-engineering-preparation/spark-warehouse


In [12]:
spark.catalog.listTables()

[Table(name='sales', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True)]

### 2.1 One database per layer
Databases are the first place to attach metadata: a **comment** and free-form **properties**. We create one database per layer of our architecture (plus a `lab` sandbox for experiments).

> Note: `owner`, `location` and `comment` are *reserved* property names in Spark, so we use `data_owner` instead.

In [13]:
layers = {
    "lab":    ("Sandbox for the managed-vs-external experiments",                                  "sandbox"),
    "bronze": ("Raw data exactly as delivered by source systems. Append-only, schema-on-read",     "bronze"),
    "silver": ("Cleaned, typed, de-duplicated and conformed data. One row per invoice line",       "silver"),
    "gold":   ("Semantic layer: business-friendly dimensions, facts and governed metrics",         "gold"),
}

for db, (comment, layer) in layers.items():
    spark.sql(f"""
        CREATE DATABASE IF NOT EXISTS {db}
        COMMENT '{comment}'
        WITH DBPROPERTIES ('layer' = '{layer}',
                           'data_owner' = 'data-engineering-class',
                           'created_by' = 'session-25-notebook')
    """)

pd.DataFrame([d._asdict() for d in spark.catalog.listDatabases()])

,name,catalog,description,locationUri
0,bronze,spark_catalog,"Raw data exactly as delivered by source systems. Append-only, schema-on-read",file:/path/to/course/spark-warehouse/bronze.db
1,default,spark_catalog,Default Hive database,file:/app/work/data-engineering-preparation/spark-warehouse
2,gold,spark_catalog,"Semantic layer: business-friendly dimensions, facts and governed metrics",file:/path/to/course/spark-warehouse/gold.db
3,lab,spark_catalog,Sandbox for the managed-vs-external experiments,file:/path/to/course/spark-warehouse/lab.db
4,silver,spark_catalog,"Cleaned, typed, de-duplicated and conformed data. One row per invoice line",file:/path/to/course/spark-warehouse/silver.db


In [14]:
# Every database gets a default location inside the warehouse: <warehouse>/<db>.db
sql("DESCRIBE DATABASE EXTENDED silver")

/usr/local/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:324: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/usr/local/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:374: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


,info_name,info_value
0,Catalog Name,spark_catalog
1,Namespace Name,silver
2,Comment,"Cleaned, typed, de-duplicated and conformed data. One row per invoice line"
3,Location,file:/path/to/course/spark-warehouse/silver.db
4,Owner,root
5,Properties,"((created_by,session-25-notebook), (data_owner,data-engineering-class), (layer,silver))"


In [15]:
spark.catalog.listDatabases()

[Database(name='bronze', catalog='spark_catalog', description='Raw data exactly as delivered by source systems. Append-only, schema-on-read', locationUri='file:/path/to/course/spark-warehouse/bronze.db'),
 Database(name='default', catalog='spark_catalog', description='Default Hive database', locationUri='file:/app/work/data-engineering-preparation/spark-warehouse'),
 Database(name='gold', catalog='spark_catalog', description='Semantic layer: business-friendly dimensions, facts and governed metrics', locationUri='file:/path/to/course/spark-warehouse/gold.db'),
 Database(name='lab', catalog='spark_catalog', description='Sandbox for the managed-vs-external experiments', locationUri='file:/path/to/course/spark-warehouse/lab.db'),
 Database(name='silver', catalog='spark_catalog', description='Cleaned, typed, de-duplicated and conformed data. One row per invoice line', locationUri='file:/path/to/course/spark-warehouse/silver.db')]

### 2.2 Where do metadata and data physically live?
* `spark-warehouse/` holds the **data** of managed tables (one folder per database, empty for now).
* `metastore_db/` holds the **metadata** — it is a Derby database, not something you edit by hand. Its internal tables (`DBS`, `TBLS`, `SDS`, `COLUMNS_V2`, `PARTITIONS`, `TABLE_PARAMS`, …) are the same in every Hive Metastore, whether on your laptop or on a production MySQL server.
* Embedded Derby allows **one** Spark process at a time — fine for class, the reason production uses a shared metastore service.

In [16]:
tree(WAREHOUSE_DIR)

metastore_dir = PROJECT_DIR / "metastore_db"
print("\nMetastore folder:", metastore_dir, "| exists:", metastore_dir.exists())
if metastore_dir.exists():
    print("contents:", sorted(p.name for p in metastore_dir.iterdir()))

spark-warehouse/
    cars1/
        part-00000-87dd588f-bbb3-4fc8-817f-de83c004ecee-c000.snappy.parquet  (12,126 bytes)

Metastore folder: /user/work/data-engineering-preparation/metastore_db | exists: True
contents: ['README_DO_NOT_TOUCH_FILES.txt', 'db.lck', 'dbex.lck', 'log', 'seg0', 'service.properties', 'tmp']


## Part 3 — Managed vs. external (unmanaged) tables

We will save the **same DataFrame** twice, once as each type, and then run four experiments.

| | Managed | External / unmanaged |
|---|---|---|
| How you create it | `saveAsTable("db.t")` or `CREATE TABLE … USING parquet` with **no** location | `saveAsTable` with `.option("path", …)`, or `CREATE TABLE … LOCATION '…'` |
| Who chooses the location | Spark (`<warehouse>/<db>.db/<table>`) | You |
| Who owns the files | Spark | You / another system |

In [17]:
lab_df = df.withColumn("invoice_month",
                       F.date_format(F.to_timestamp("InvoiceDate", "M/d/yyyy H:mm"), "yyyy-MM"))

# MANAGED: we only give Spark a NAME. Spark decides where the files go.
(lab_df.write
       .mode("overwrite")
       .format("parquet")
       .partitionBy("invoice_month")
       .saveAsTable("lab.sales_managed"))

# EXTERNAL: same data, but WE decide where the files go.
external_path = EXTERNAL_DIR / "lab" / "sales_external"
(lab_df.write
       .mode("overwrite")
       .format("parquet")
       .partitionBy("invoice_month")
       .option("path", external_path.as_posix())     # <- this one option makes it EXTERNAL
       .saveAsTable("lab.sales_external"))

catalog_tables("lab")

26/11/10 15:01:16 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.
26/11/10 15:01:16 WARN HiveConf: HiveConf of name hive.internal.ss.authz.settings.applied.marker does not exist


,name,catalog,namespace,description,tableType,isTemporary
0,sales_external,spark_catalog,[lab],None,EXTERNAL,False
1,sales_managed,spark_catalog,[lab],None,MANAGED,False


In [18]:
# Same data, same schema — the metastore records a different Type and Location
pd.concat({t: table_summary(f"lab.{t}", keys=("Type", "Provider", "Location", "Partition Provider"))["value"]
           for t in ["sales_managed", "sales_external"]}, axis=1)

,sales_managed,sales_external
property,,
Type,MANAGED,EXTERNAL
Provider,parquet,parquet
Location,file:/path/to/course/spark-warehouse/lab.db/sales_managed,file:/user/work/data-engineering-preparation/external-data/lab/sales_external
Partition Provider,Catalog,Catalog


In [19]:
managed_path  = table_location("lab.sales_managed")
external_path = table_location("lab.sales_external")

print("MANAGED table files (inside the warehouse):")
tree(managed_path)
print("\nEXTERNAL table files (outside the warehouse, in a folder we chose):")
tree(external_path)

MANAGED table files (inside the warehouse):
sales_managed/
    invoice_month=2010-12/
        part-00000-3d028be7-c2c4-4bc1-a20e-a42e9a471040.c000.snappy.parquet  (18,399 bytes)
    invoice_month=2011-01/
        part-00000-3d028be7-c2c4-4bc1-a20e-a42e9a471040.c000.snappy.parquet  (16,232 bytes)
    invoice_month=2011-02/
        part-00000-3d028be7-c2c4-4bc1-a20e-a42e9a471040.c000.snappy.parquet  (14,306 bytes)
    invoice_month=2011-03/
        part-00000-3d028be7-c2c4-4bc1-a20e-a42e9a471040.c000.snappy.parquet  (17,903 bytes)
    invoice_month=2011-04/
        part-00000-3d028be7-c2c4-4bc1-a20e-a42e9a471040.c000.snappy.parquet  (16,082 bytes)
    invoice_month=2011-05/
        part-00000-3d028be7-c2c4-4bc1-a20e-a42e9a471040.c000.snappy.parquet  (18,218 bytes)
    invoice_month=2011-06/
        part-00000-3d028be7-c2c4-4bc1-a20e-a42e9a471040.c000.snappy.parquet  (16,620 bytes)
    invoice_month=2011-07/
        part-00000-3d028be7-c2c4-4bc1-a20e-a42e9a471040.c000.snappy.parquet  (18,

In [20]:
spark.catalog.listTables('lab')

[Table(name='sales_external', catalog='spark_catalog', namespace=['lab'], description=None, tableType='EXTERNAL', isTemporary=False),
 Table(name='sales_managed', catalog='spark_catalog', namespace=['lab'], description=None, tableType='MANAGED', isTemporary=False),
 Table(name='sales', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True)]

### Some differences


| Operation | Managed table | External table |
|---|---|---|
| `DROP TABLE` | deletes metadata **and** files | deletes metadata only — files remain |
| `TRUNCATE TABLE` | deletes the files' contents | refused |
| `ALTER TABLE … RENAME` | renames and **moves** the folder | renames the metadata only |
| `INSERT` / `INSERT OVERWRITE` | ✔ | ✔ (you are still writing into your own folder) |
| Typical use | data the platform produces and owns: Silver, Gold aggregates | data owned by someone else or shared with other engines: raw landing files, data lake folders read by other tools |
| Main risk | accidental `DROP` = data loss | orphaned files, stale partition metadata |

> In Databricks / Unity Catalog the same split exists (managed vs. external tables, with extra governance on top), and Delta/Iceberg tables add a transaction log, but the ownership idea is identical.

In [21]:
spark.stop()